# CASE 08 — Square 8×8 + 5×5 — Convolution Center

In [1]:
%%writefile case08_sq_5x5_convCenter.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define WIDTH       8
#define HEIGHT      8
#define MASK_WIDTH  5
#define MASK_HEIGHT 5
#define BLOCK_SIZE  8

__global__ void convolutionCenter(int *dA, int *dMask, int *dC,
                                  int width, int height,
                                  int mWidth, int mHeight)
{
    int col = threadIdx.x + blockIdx.x * blockDim.x;
    int row = threadIdx.y + blockIdx.y * blockDim.y;
    if (row < height && col < width)
    {
        int sum = 0;
        for (int i = 0; i < mHeight; i++)
            for (int j = 0; j < mWidth; j++)
            {
                int r = row + i - mHeight/2;   /* center */
                int c = col + j - mWidth/2;
                if (r>=0 && r<height && c>=0 && c<width)
                    sum += dA[r*width+c] *
                           dMask[(mHeight-1-i)*mWidth+(mWidth-1-j)]; /* flipped */
            }
        dC[row*width+col] = sum;
    }
}

void printMatrix(const char *label, int *M, int w, int h)
{
    printf("\n%s:\n", label);
    for (int r = 0; r < h; r++)
    {
        for (int c = 0; c < w; c++)
            printf("%6d", M[r*w+c]);
        printf("\n");
    }
}

int main()
{
    int size     = WIDTH * HEIGHT * sizeof(int);
    int maskSize = MASK_WIDTH * MASK_HEIGHT * sizeof(int);

    int *hA    = (int*) malloc(size);
    int *hMask = (int*) malloc(maskSize);
    int *hC    = (int*) malloc(size);

    srand(time(NULL));
    for (int i = 0; i < WIDTH*HEIGHT; i++)
        hA[i] = rand()%9+1;

    int tempMask[5][5] = {
        {1,1,1,1,1},
        {1,2,2,2,1},
        {1,2,4,2,1},
        {1,2,2,2,1},
        {1,1,1,1,1}
    };
    for (int i = 0; i < MASK_HEIGHT; i++)
        for (int j = 0; j < MASK_WIDTH; j++)
            hMask[i*MASK_WIDTH+j] = tempMask[i][j];

    printMatrix("Input Matrix (8x8)", hA,    WIDTH,      HEIGHT);
    printMatrix("Mask (5x5)",         hMask, MASK_WIDTH, MASK_HEIGHT);

    int *dA, *dMask, *dC;
    cudaMalloc((void**)&dA,    size);
    cudaMalloc((void**)&dMask, maskSize);
    cudaMalloc((void**)&dC,    size);
    cudaMemcpy(dA,    hA,    size,     cudaMemcpyHostToDevice);
    cudaMemcpy(dMask, hMask, maskSize, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 DimGrid((int)ceil((float)WIDTH/BLOCK_SIZE),
                 (int)ceil((float)HEIGHT/BLOCK_SIZE), 1);

    cudaEvent_t start, stop; float gpuTime;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    convolutionCenter<<<DimGrid,DimBlock>>>(dA,dMask,dC,
                       WIDTH,HEIGHT,MASK_WIDTH,MASK_HEIGHT);

    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpuTime, start, stop);
    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    printMatrix("OUTPUT: Convolution Center (8x8, 5x5)", hC, WIDTH, HEIGHT);
    printf("\nGPU Time: %.4f ms\n", gpuTime);
    printf("Grid: %dx%d  Block: %dx%d\n",
            DimGrid.x,DimGrid.y,DimBlock.x,DimBlock.y);

    cudaFree(dA); cudaFree(dMask); cudaFree(dC);
    free(hA); free(hMask); free(hC);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}

Writing case08_sq_5x5_convCenter.cu


In [2]:
!nvcc -arch=sm_75 case07_sq_5x5_convTopLeft.cu -o case07_sq_5x5_convTopLeft

!./case07_sq_5x5_convTopLeft


Input Matrix (8x8):
     5     1     9     1     1     9     4     9
     9     2     2     2     5     3     7     3
     1     6     7     8     8     3     4     9
     5     2     3     6     5     3     6     9
     3     3     7     2     2     1     8     1
     1     7     9     3     9     4     6     7
     9     3     5     5     3     8     5     7
     9     5     3     2     8     6     9     1

Mask (5x5):
     1     1     1     1     1
     1     2     2     2     1
     1     2     4     2     1
     1     2     2     2     1
     1     1     1     1     1

OUTPUT: Convolution TopLeft (8x8, 5x5):
   157   163   175   169   155   135    81    31
   167   172   171   170   159   129    79    29
   178   172   169   178   169   120    79    33
   181   165   183   180   167   129    74    25
   150   152   159   171   148   112    59    16
   110   110   133   134   127    77    43    15
    62    61    70    77    63    46    23     8
    27    24    28    26    24    1